# APRR Extension: CROW + OctoRoute Routing Benchmark

**Paper:** *Extending APRR with Chain-of-Reasoning Over Workload (CROW) and Octopus-Inspired Distributed Routing (OctoRoute)*  
**Author:** Jenisha T — PhD Candidate, MS Ramaiah University, Bengaluru  
**Repo:** [github.com/joyjeni/aprr-multi-agent-routing](https://github.com/joyjeni/aprr-multi-agent-routing)

---

## Overview

This notebook benchmarks three routing strategies on ToolBench-style synthetic queries:

| Router | Paradigm | Key Mechanism |
|---|---|---|
| **APRR** | RL policy iteration | W-matrix + success-weighted ΔW + decay |
| **CROW** | CoT-augmented routing | Complexity-gated deliberation, reasoning-quality ΔW |
| **OctoRoute** | Bio-inspired dispatch | Functional tokens + chromatophore signals + arm-local W |
| **CROW-OctoRoute** | Hybrid | Arm dispatch → CROW deliberation within arm |

**Research question:** Does CoT-style deliberation (CROW) or distributed arm-based routing (OctoRoute) improve over APRR's scalar reinforcement signal on ToolBench G1/G2/G3?


In [ ]:
# Install dependencies
!pip install -q numpy matplotlib seaborn pandas

In [ ]:
# Clone the repo to get the extension modules
!git clone --quiet https://github.com/joyjeni/aprr-multi-agent-routing.git
import sys
sys.path.insert(0, 'aprr-multi-agent-routing/src')
sys.path.insert(0, 'aprr-multi-agent-routing/src/aprr_extensions')
print('Repo loaded')

In [ ]:
from aprr_extensions import CROWRouter, OctoArm, OctoRouteRouter
print('CROWRouter:', CROWRouter)
print('OctoRouteRouter:', OctoRouteRouter)

In [ ]:
# Run the full benchmark
import subprocess
result = subprocess.run(
    ['python3', 'aprr-multi-agent-routing/experiments/extensions/aprr_comparison_benchmark.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

In [ ]:
import json, pandas as pd

with open('aprr-multi-agent-routing/results/extension_results.json') as f:
    results = json.load(f)

df = pd.DataFrame([
    {
        'Router': name,
        'Success Rate': round(v['avg_success'], 3),
        'Latency (ms)': round(v['avg_latency_ms'], 1),
        'Avg Hops': round(v['avg_hops'], 2),
        'Interpretability': v['interpretability'],
        'Overhead': v['overhead'],
    }
    for name, v in results['routers'].items()
])
df = df.sort_values('Success Rate', ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

routers = df['Router'].tolist()
success = df['Success Rate'].tolist()
latency = df['Latency (ms)'].tolist()

COLORS = {
    'APRR': '#1f77b4',
    'CROW': '#ff7f0e',
    'OctoRoute': '#2ca02c',
    'CROW-OctoRoute': '#9467bd',
    'StaticSemantic': '#d62728',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('APRR Extension: CROW vs OctoRoute vs Baselines', fontsize=14, fontweight='bold')

colors = [COLORS.get(r, '#aaa') for r in routers]

# Fig 1: Success Rate
axes[0].barh(routers, success, color=colors)
axes[0].set_xlabel('Success Rate')
axes[0].set_title('Success Rate (higher = better)')
axes[0].set_xlim(0, 1)
for i, v in enumerate(success):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

# Fig 2: Latency
axes[1].barh(routers, latency, color=colors)
axes[1].set_xlabel('Latency (ms)')
axes[1].set_title('Avg Latency (lower = better)')
for i, v in enumerate(latency):
    axes[1].text(v + 2, i, f'{v:.0f}ms', va='center', fontsize=9)

# Fig 3: Pareto scatter (Success vs Latency)
for r, s, l in zip(routers, success, latency):
    axes[2].scatter(l, s, color=COLORS.get(r, '#aaa'), s=120, zorder=5)
    axes[2].annotate(r, (l, s), textcoords='offset points', xytext=(5, 3), fontsize=8)
axes[2].set_xlabel('Latency (ms)')
axes[2].set_ylabel('Success Rate')
axes[2].set_title('Pareto Frontier (top-left = best)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('aprr_extension_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Single-query walkthrough: see CROW deliberation in action
import numpy as np

AGENTS = ['WeatherAgent', 'MarketAgent', 'CropAgent', 'SchemeAgent', 'SoilAgent']

crow = CROWRouter(
    agents=AGENTS,
    thought_budget_threshold=0.3,  # lower = more deliberation
    max_deliberation_rounds=3
)

test_queries = [
    'What is the current price of tomatoes in Chennai mandi?',
    'Rainfall forecast for Coimbatore district this week.',
    'Which PM scheme covers crop insurance for paddy farmers in Tamil Nadu?',
    'Compare soil nitrogen levels for sugarcane vs rice cultivation.',
]

print('CROW Routing Walkthrough')
print('=' * 70)
for q in test_queries:
    result = crow.route(q)
    print(f"Query  : {q[:60]}")
    print(f"Agent  : {result['agent_name']} (idx={result['agent']})")
    print(f"Mode   : {result['mode']} | Complexity: {result['complexity_score']}")
    if result.get('reasoning'):
        print(f"Reason : {result['reasoning'].get('rationale', '')[:80]}")
    print()

In [ ]:
# OctoRoute walkthrough: see arm dispatch via functional tokens
ARM_DOMAINS = ['weather', 'market_prices', 'crop_advisory', 'schemes', 'soil_health']

octo = OctoRouteRouter(
    agents=AGENTS,
    arm_domains=ARM_DOMAINS,
    chromatophore_threshold=0.05
)

print('OctoRoute Functional Token Dispatch')
print('=' * 70)
for q in test_queries:
    result = octo.route(q)
    arm_name = ARM_DOMAINS[result['arm']]
    print(f"Query  : {q[:60]}")
    print(f"Token  : {result['functional_token']} → Arm: {arm_name}")
    print(f"Agent  : {AGENTS[result['agent']]} (idx={result['agent']})")
    sigs = result.get('chromatophore_signals', [])
    sig_str = ' | '.join(f"{ARM_DOMAINS[s['arm_id']]}:{s['signal']}({s['domain_match']:.2f})" for s in sigs[:3])
    print(f"Signals: {sig_str}")
    print()

print('\nArm Specialisation Index (ASI):')
asi = octo.get_arm_specialisation_index()
for k, v in asi.items():
    print(f"  {k}: {v:.4f}")

## Key Findings

| Router | Success | Latency | Best For |
|---|---|---|---|
| APRR | Baseline | ~270ms | General-purpose balanced routing |
| CROW | +0–2% on complex queries | ~470ms | Multi-step reasoning tasks needing interpretability |
| OctoRoute | Comparable, faster | ~210ms | Domain-clear, latency-sensitive edge deployment |
| CROW-OctoRoute | Best complex-query success | ~500ms | Research demonstrator; interpretable + domain-aware |

**Hypothesis confirmed:** OctoRoute achieves **~22% latency reduction** over APRR via direct functional-token dispatch, at parity success rate. CROW adds interpretable reasoning traces with **~74% higher latency** but is expected to show larger gains on G3 (out-of-distribution) ToolBench splits.

**Next steps:**
1. Run on real ToolBench G1/G2/G3 with actual LLM inference
2. Integrate CROW traces into SessionRerank (Objective 1) as priority signals
3. Wire OctoRoute chromatophore signals into MNCD distress broadcast (Objective 3)
4. Use arm dispatch for domain-gated FCNP pruning (Objective 4)
